# Variant Effect Map (VEM)

Two plots: TANGO (Δ Aggregation) and PASTA (Δ Best Energy).

In [ ]:
import pandas as pd
import numpy as np
import io
import re
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyBboxPatch
from matplotlib.gridspec import GridSpec

# ── Palette ──────────────────────────────────────────────────
BG = "#0a0c14"
SURFACE = "#111520"
MUTED = "#64748b"
TEXT = "#e2e8f0"
WT_COL = "#4ff7c0"

# ── Wildtype Aβ42 sequence ────────────────────────────────────
WT_SEQ = "DAEFRHDSGYEVHHQKLVFFAEDVGSNKGAIIGLMVGGVVIA"
AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")

In [ ]:
# 1. Load data


def load_tango(path="../predictions/tango/tango_input_aggregation.txt"):
    df = pd.read_csv(path, sep="\t")
    wt = df[df["Sequence"] == "Wildtype_Abeta_42"]["Aggregation"].iloc[0]
    records = []
    for _, row in df.iterrows():
        name = row["Sequence"]
        if "Wildtype" in name:
            continue
        m = re.match(r"^([A-Z])(\d+)([A-Z])", name)
        if not m:
            continue
        records.append(
            {
                "pos": int(m.group(2)),
                "wt_aa": m.group(1),
                "mut_aa": m.group(3),
                "delta": row["Aggregation"] - wt,
            }
        )
    return pd.DataFrame(records)


def load_pasta(path="../predictions/pasta/pasta.csv"):
    with open(path, encoding="utf-8-sig") as f:
        df = pd.read_csv(io.StringIO(f.read().replace(",", ".")), sep=";")
    wt = df[df["Protein name"].str.contains("Wildtype")]["Best Energy"].iloc[0]
    records = []
    for _, row in df.iterrows():
        name = row["Protein name"]
        if "Wildtype" in name:
            continue
        m = re.match(r"^([A-Z])(\d+)([A-Z])", name)
        if not m:
            continue
        records.append(
            {
                "pos": int(m.group(2)),
                "wt_aa": m.group(1),
                "mut_aa": m.group(3),
                "delta": row["Best Energy"] - wt,
            }
        )
    return pd.DataFrame(records)


tango_df = load_tango()
pasta_df = load_pasta()

In [ ]:
# 2. Position × amino acid matrix


def build_matrix(df):
    positions = sorted(df["pos"].unique())
    mat = pd.DataFrame(np.nan, index=AA_ORDER, columns=positions)
    for _, row in df.iterrows():
        if row["mut_aa"] in AA_ORDER:
            mat.loc[row["mut_aa"], row["pos"]] = row["delta"]
    return mat


tango_mat = build_matrix(tango_df)
pasta_mat = build_matrix(pasta_df)

In [ ]:
# 3. Plotting functions


def draw_vem(ax, mat, title, unit_label, cmap_name="RdBu_r"):
    ax.set_facecolor(SURFACE)

    positions = sorted(mat.columns)
    n_pos = len(positions)
    n_aa = len(AA_ORDER)

    vals = mat.values[~np.isnan(mat.values)]
    vlim = np.max(np.abs(vals)) * 1.02
    norm = mcolors.TwoSlopeNorm(vmin=-vlim, vcenter=0, vmax=vlim)
    cmap = matplotlib.colormaps[cmap_name]

    cell_w, cell_h = 0.88, 0.88

    for i, pos in enumerate(positions):
        bg = "#161b2e" if i % 2 == 0 else SURFACE
        ax.axvspan(i - 0.5, i + 0.5, color=bg, alpha=0.5, zorder=0)

    for j, aa in enumerate(AA_ORDER):
        for i, pos in enumerate(positions):
            val = mat.loc[aa, pos]
            wt_here = WT_SEQ[pos - 1] if pos <= len(WT_SEQ) else "?"

            if aa == wt_here:
                ax.scatter(
                    i,
                    j,
                    marker="D",
                    s=18,
                    color=WT_COL,
                    alpha=0.6,
                    zorder=4,
                    linewidths=0,
                )
                continue

            if np.isnan(val):
                rect = FancyBboxPatch(
                    (i - cell_w / 2, j - cell_h / 2),
                    cell_w,
                    cell_h,
                    boxstyle="round,pad=0.04",
                    facecolor="#0d1120",
                    linewidth=0,
                    zorder=2,
                    alpha=0.4,
                )
                ax.add_patch(rect)
                continue

            rect = FancyBboxPatch(
                (i - cell_w / 2, j - cell_h / 2),
                cell_w,
                cell_h,
                boxstyle="round,pad=0.04",
                facecolor=cmap(norm(val)),
                linewidth=0,
                zorder=2,
            )
            ax.add_patch(rect)

            if abs(val) > np.percentile(np.abs(vals[vals != 0]), 85):
                tc = "white" if abs(val) > 0.5 * vlim else TEXT
                fs = 4.5 if abs(val) > 200 else 4.0
                ax.text(
                    i,
                    j,
                    f"{val:+.0f}",
                    ha="center",
                    va="center",
                    fontsize=fs,
                    fontfamily="monospace",
                    color=tc,
                    fontweight="bold",
                    zorder=5,
                )

    ax.set_xlim(-0.6, n_pos - 0.4)
    ax.set_ylim(-0.6, n_aa - 0.4)
    ax.set_xticks(range(n_pos))
    ax.set_xticklabels(
        [f"{p}\n{WT_SEQ[p - 1]}" for p in positions],
        fontsize=6,
        fontfamily="monospace",
        color=MUTED,
    )
    ax.tick_params(axis="x", length=0, pad=2)
    ax.set_yticks(range(n_aa))
    ax.set_yticklabels(AA_ORDER, fontsize=7, fontfamily="monospace", color=TEXT)
    ax.tick_params(axis="y", length=0, pad=4)
    for sp in ax.spines.values():
        sp.set_visible(False)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(
        sm, ax=ax, shrink=0.7, pad=0.012, aspect=28, orientation="vertical"
    )
    cbar.set_label(
        unit_label, color=MUTED, fontsize=8, fontfamily="monospace", labelpad=8
    )
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=MUTED, fontsize=7)
    cbar.outline.set_edgecolor("#1e2540")

    ax.set_title(
        title,
        color="white",
        fontsize=11,
        fontfamily="monospace",
        fontweight="bold",
        pad=10,
    )

    wt_marker = plt.scatter(
        [], [], marker="D", s=30, color=WT_COL, alpha=0.7, label="WT residue"
    )
    ax.legend(
        handles=[wt_marker],
        loc="upper left",
        frameon=True,
        framealpha=0.2,
        edgecolor=MUTED,
        facecolor=BG,
        fontsize=7,
        labelcolor=TEXT,
    )


def draw_position_profile(ax, df, title, ylabel, color_pos, color_neg):
    ax.set_facecolor(SURFACE)
    positions = sorted(df["pos"].unique())
    means, mins_, maxs_ = [], [], []
    for p in positions:
        v = df[df["pos"] == p]["delta"].values
        means.append(np.mean(v))
        mins_.append(np.min(v))
        maxs_.append(np.max(v))

    x = np.arange(len(positions))
    mn = np.array(means)
    lo = np.array(mins_)
    hi = np.array(maxs_)

    ax.fill_between(x, lo, hi, color="white", alpha=0.06, zorder=1)
    ax.plot(x, mn, color="white", lw=0.6, alpha=0.3, zorder=2)
    for i in range(len(x)):
        c = color_pos if mn[i] >= 0 else color_neg
        ax.vlines(x[i], lo[i], hi[i], color=c, lw=0.6, alpha=0.35, zorder=2)
        ax.scatter(x[i], mn[i], color=c, s=22, zorder=3, linewidths=0)

    ax.axhline(0, color=WT_COL, lw=0.8, linestyle="--", alpha=0.4)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{p}\n{WT_SEQ[p - 1]}" for p in positions],
        fontsize=5.5,
        fontfamily="monospace",
        color=MUTED,
    )
    ax.tick_params(axis="x", length=0, pad=2)
    ax.tick_params(axis="y", colors=MUTED, labelsize=7)
    for sp in ax.spines.values():
        sp.set_edgecolor("#1e2540")
    ax.set_title(
        title,
        color="white",
        fontsize=9,
        fontfamily="monospace",
        fontweight="bold",
        pad=8,
    )
    ax.set_ylabel(ylabel, color=MUTED, fontsize=8, fontfamily="monospace", labelpad=6)
    ax.set_xlabel(
        "Position  (pos / WT residue)",
        color=MUTED,
        fontsize=7.5,
        fontfamily="monospace",
        labelpad=4,
    )

In [ ]:
# 4. Figure 1 — TANGO

import matplotlib

fig1 = plt.figure(figsize=(22, 10), facecolor=BG)
gs1 = GridSpec(2, 1, figure=fig1, hspace=0.5, height_ratios=[1, 0.38])

draw_vem(
    fig1.add_subplot(gs1[0]),
    tango_mat,
    title="TANGO  ·  Variant Effect Map  ·  Δ Aggregation Score vs Wildtype",
    unit_label="Δ Aggregation\n(red = higher than WT)",
    cmap_name="RdBu_r",
)

draw_position_profile(
    fig1.add_subplot(gs1[1]),
    tango_df,
    title="TANGO  ·  Δ per position  (mean ± range)",
    ylabel="Δ Aggregation",
    color_pos="#ef4444",
    color_neg="#3b82f6",
)

fig1.suptitle(
    "Variant Effect Map  ·  Aβ42  ·  TANGO\n"
    "Red = increased aggregation  ·  Blue = decreased  ·  ◆ = WT residue",
    color="white",
    fontsize=13,
    fontfamily="monospace",
    fontweight="bold",
    y=1.01,
)

plt.savefig(
    "vem_tango.png", dpi=180, bbox_inches="tight", facecolor=BG, edgecolor="none"
)
print("Saved: vem_tango.png")
plt.close()

In [ ]:
# 5. Figure 2 — PASTA

fig2 = plt.figure(figsize=(22, 10), facecolor=BG)
gs2 = GridSpec(2, 1, figure=fig2, hspace=0.5, height_ratios=[1, 0.38])

draw_vem(
    fig2.add_subplot(gs2[0]),
    pasta_mat,
    title="PASTA 2.0  ·  Variant Effect Map  ·  Δ Best Energy vs Wildtype",
    unit_label="Δ Best Energy (kcal/mol)\n(blue = more stable than WT)",
    cmap_name="RdBu",
)

draw_position_profile(
    fig2.add_subplot(gs2[1]),
    pasta_df,
    title="PASTA  ·  Δ Best Energy per position  (mean ± range)",
    ylabel="Δ Best Energy (kcal/mol)",
    color_pos="#3b82f6",
    color_neg="#ef4444",
)

fig2.suptitle(
    "Variant Effect Map  ·  Aβ42  ·  PASTA 2.0\n"
    "Blue = more stable than WT (destabilizes β-sheet)  ·  Red = less stable  ·  ◆ = WT residue",
    color="white",
    fontsize=13,
    fontfamily="monospace",
    fontweight="bold",
    y=1.01,
)

plt.savefig(
    "vem_pasta.png", dpi=180, bbox_inches="tight", facecolor=BG, edgecolor="none"
)
print("Saved: vem_pasta.png")
plt.close()